In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Identify Net-New Field Sales Leads with Places Insights

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fplaces_insights%2Fnotebooks%2Fidentify_sales_leads%2Fplaces_insights_sales_leads.ipynb?utm_source=places_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/identify_sales_leads/places_insights_sales_leads.ipynb&utm_source=places_insights_notebooks">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/identify_sales_leads/places_insights_sales_leads.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<br><br><br>

> **⚠️ Important Requirement:** To run the queries in this notebook, your Google Cloud Project must have access to the **US Places Insights dataset**. For instructions on how to request and configure access, please see [Set up Places Insights](https://developers.google.com/maps/documentation/placesinsights/cloud-setup).

### Overall Goal

This notebook demonstrates an end-to-end data pipeline based on the [Places Insights Sales Leads Architecture](https://developers.google.com/maps/architecture/places-insights-sales-leads). It solves a common, high-value business problem: **finding every operational target business in a specific territory that is *not* already in your CRM.**

By visualizing commercial density and using spatial joins, this workflow transforms raw map data into a clean, actionable list of highly targeted sales prospects.

### Key Technologies Used

*   **[Places Insights](https://developers.google.com/maps/documentation/placesinsights):** To provide the underlying commercial datasets and geospatial count functions.
*   **[BigQuery](https://cloud.google.com/bigquery):** To execute analytical queries, handle complex polygons, and perform the final `LEFT JOIN` exclusion.
*   **[Address Validation API](https://developers.google.com/maps/documentation/address-validation):** To confidently translate unstructured CRM addresses into definitive `place_id`s.
*   **[Places API (Text Search)](https://developers.google.com/maps/documentation/places/web-service/text-search):** To serve as a highly accurate fallback for extracting establishment IDs.
*   **[Google Maps 2D Tiles](https://developers.google.com/maps/documentation/tile/2d-tiles-overview):** To visualize the territory density on an authentic Google basemap.
*   **Python Libraries:** **GeoPandas** (spatial manipulation), **Folium** (interactive mapping), and **Requests** (API calls).

*Note: This notebook executes queries and API calls that incur costs. See [Google Maps Platform Pricing](https://mapsplatform.google.com/pricing/) and [BigQuery Pricing](https://cloud.google.com/bigquery/pricing) for details.*

### The Step-by-Step Workflow

1.  **Define Target Place Types:** In a production environment, your first step is mapping your ideal customer profile to specific [Places Insights Place Types](https://developers.google.com/maps/architecture/places-insights-sales-leads#step_1_define_your_target_place_types). *For this notebook, we have pre-selected `['restaurant', 'bar', 'cafe', 'coffee_shop']` to focus on the food and beverage sector.*
2.  **Extract High-Potential Areas:** We use the `PLACES_COUNT_PER_H3` function combined with a public city boundary dataset to identify the most densely packed commercial zones in New York City, visualising the results on an interactive heatmap.
3.  **Normalize CRM Data:** We take a sample of raw, unstructured CRM addresses and pass them through a two-step API pipeline (Address Validation & Text Search) to extract the definitive Google Maps `place_id` for every existing customer.
4.  **Whitespace Exclusion Analysis:** We zoom in on the highest-density territory (Union Square) and run a BigQuery spatial query. By cross-referencing the businesses in the territory against our newly normalized CRM IDs, we filter out existing customers to reveal a list of 100% net-new sales leads.

### How to Use This Notebook

1.  **Prerequisites & Secrets:** Before running this notebook, you must configure two environment variables in the Colab "Secrets" tab (the **key icon** on the left menu):
    *   `GCP_PROJECT_ID`: Your Google Cloud Project ID. **Crucially, this project must be authorized to access the US Places Insights dataset (see [Set up Places Insights](https://developers.google.com/maps/documentation/placesinsights/cloud-setup)).**
    *   `GMP_API_KEY`: A Google Maps Platform API key with the following APIs enabled: *Address Validation API, Places API (New), and Map Tiles API*.
2.  **Authentication:** The first cell will prompt you to authenticate your Google Account. Ensure the account you use has BigQuery Data Viewer/Job User permissions for your Project ID.
3.  **Run the Cells:** Once authenticated, execute the cells in order from top to bottom to extract, visualize, and generate your leads.

In [ ]:
# @title 1. Setup & Authentication
# @markdown Authenticate to Google Cloud, retrieve secrets, and initialize the BigQuery client.
import sys
import requests
import pandas as pd
import geopandas as gpd
from shapely import wkt
from google.cloud import bigquery
import google.auth
import getpass

try:
    # --- ATTEMPT 1: Standard Consumer Colab ---
    from google.colab import userdata, auth

    # Attempt to retrieve secrets from the left-hand sidebar
    GCP_PROJECT_ID = userdata.get('GCP_PROJECT_ID')
    GMP_API_KEY = userdata.get('GMP_API_KEY')
    print("✅ Standard Colab Secrets found.")

    # Authenticate via user pop-up
    print("🔄 Authenticating user...")
    auth.authenticate_user(project_id=GCP_PROJECT_ID)

    # Initialize BigQuery client
    client = bigquery.Client(project=GCP_PROJECT_ID)
    print(f"✅ Authenticated to Google Cloud Project: {GCP_PROJECT_ID}")

except (ImportError, Exception) as e:
    # --- ATTEMPT 2: Colab Enterprise / Local Jupyter ---
    print(f"ℹ️ Standard Colab setup skipped ({e}). Falling back to Enterprise/Local Auth...")

    # 1. Retrieve Environment Authentication & Project ID automatically
    credentials, GCP_PROJECT_ID = google.auth.default()
    print(f"✅ Authenticated via default credentials to Project: {GCP_PROJECT_ID}")

    # 2. Securely input the Google Maps API Key
    print("🔑 Please paste your Google Maps Platform API Key and press Enter:")
    GMP_API_KEY = getpass.getpass()

    # 3. Initialize BigQuery client with environment credentials
    client = bigquery.Client(credentials=credentials, project=GCP_PROJECT_ID)

print("✅ BigQuery Client Initialized.")

In [ ]:
# @title 2. Backend Initialization: Session, Copyright & Assets
# @markdown This cell manages the API handshake for the Google Maps 2D Tiles.
# @markdown 1.  **Session Creation:** Authenticates and requests a "Roadmap" session.
# @markdown 2.  **Attribution Fetching:** Queries the API for copyright text for the selected city.
# @markdown 3.  **Asset Preparation:** Generates the HTML for the Google Maps logo overlay.

# Center of New York City for the Maps API Tile generation
CITY_CENTER_LAT = 40.7128
CITY_CENTER_LNG = -74.0060

# --- 1. Create Google Maps Session ---
print("🗺️ Initializing Google Maps Session...")
session_url = f"https://tile.googleapis.com/v1/createSession?key={GMP_API_KEY}"
headers = {"Content-Type": "application/json"}
payload = {
    "mapType": "roadmap",
    "language": "en-US",
    "region": "US"
}

try:
    response = requests.post(session_url, json=payload, headers=headers)
    response.raise_for_status()
    session_token = response.json().get("session")
    print(f"✅ Session Token acquired: {session_token[:10]}...")
except Exception as e:
    raise RuntimeError(f"Failed to initialize Google Maps session: {e}")

# --- 2. Fetch Dynamic Attribution ---
# We calculate a small dynamic bounding box around the chosen City Center
# to ensure we fetch the correct copyright for that specific region.
delta = 0.05
north, south = CITY_CENTER_LAT + delta, CITY_CENTER_LAT - delta
east, west = CITY_CENTER_LNG + delta, CITY_CENTER_LNG - delta
zoom_level_attr = 13

viewport_url = (
    f"https://tile.googleapis.com/tile/v1/viewport?key={GMP_API_KEY}"
    f"&session={session_token}"
    f"&zoom={zoom_level_attr}"
    f"&north={north}&south={south}"
    f"&west={west}&east={east}"
)

try:
    vp_response = requests.get(viewport_url)
    vp_response.raise_for_status()
    google_attribution = vp_response.json().get('copyright', 'Map data © Google')
    print("✅ Attribution fetched.")
except Exception as e:
    print(f"⚠️ Warning: Could not fetch attribution ({e}). Defaulting.")
    google_attribution = "Map data © Google"

# --- 3. Construct Assets & Helpers ---
tiles_template_url = f"https://tile.googleapis.com/v1/2dtiles/{{z}}/{{x}}/{{y}}?session={session_token}&key={GMP_API_KEY}"
logo_url = "https://maps.gstatic.com/mapfiles/api-3/images/google_white3.png"

logo_html = f"""



"""
print("✅ Map Assets prepared.")

### Extract Business Counts to Identify High-Potential Areas

Before extracting individual business leads, it is highly strategic to figure out *where* the highest concentration of your target audience is located. This ensures your field sales team focuses their energy on the most target-rich environments.

As detailed in the [Places Insights Sales Leads Architecture](https://developers.google.com/maps/architecture/places-insights-sales-leads#step_2_extract_business_counts_to_identify_high-potential_areas), we use the `PLACES_COUNT_PER_H3` function to aggregate businesses into a uniform hexagonal grid.

In the query below, we will perform a powerful data-engineering trick:
1. **Dynamic Boundaries:** Instead of manually drawing a polygon, we use BigQuery scripting (`DECLARE` and `SET`) to instantly fetch the exact city limits of New York City from the public `overture_maps` dataset.
2. **Density Calculation:** We pass that complex NYC polygon into Places Insights to count every operational restaurant, bar, and cafe within it.
3. **Prioritization:** By sorting the output by `count DESC`, the absolute hottest commercial zones in the city will automatically float to the top of our Pandas DataFrame.

In [ ]:
# @title 3. Find High-Potential Areas (NYC Example)
# @markdown Dynamically fetch the NYC polygon and query Places Insights for the densest H3 cells.

# The query provided from the architecture document
# Note: I swapped 'ORDER BY count' to 'ORDER BY place_count' to strictly match the V2 function schema.
nyc_density_query = f"""
DECLARE geo GEOGRAPHY;

-- Get the geography for New York City using the Overture Maps public dataset
SET geo = (SELECT geometry FROM `bigquery-public-data.overture_maps.division_area`
  WHERE country = 'US' AND subtype = 'locality' AND names.primary = 'New York' LIMIT 1);

SELECT *
FROM `{GCP_PROJECT_ID}.places_insights___us.PLACES_COUNT_PER_H3`(
  JSON_OBJECT(
      'geography', geo,
      'h3_resolution', 8,
      'types',['restaurant', 'bar', 'cafe', 'coffee_shop'],
      'business_status',['OPERATIONAL']
  )
)
ORDER BY count DESC;
"""

print("🚀 Running BigQuery script to fetch NYC geography and calculate H3 densities...")

try:
    # client.query() seamlessly handles multi-statement BigQuery scripts
    job = client.query(nyc_density_query)

    # Convert the results to a Pandas DataFrame
    nyc_h3_df = job.to_dataframe()

    print(f"✅ Successfully analyzed {len(nyc_h3_df)} H3 cells in New York City.")
    display(nyc_h3_df.head())

except Exception as e:
    print(f"❌ Query failed: {e}")

In [ ]:
# @title 4. Visualization: NYC H3 Density Heatmap
# @markdown Renders the Places Insights H3 grid as a Choropleth map over Google Maps 2D Tiles.
# @markdown
# @markdown **Strategic Insight:** By visualizing the data, the deep red clusters instantly reveal the city's commercial hotspots. Notice the concentration of target businesses (restaurants, bars, cafes) around **Union Square**. Because of this high density, we will focus our whitespace exclusion analysis strictly on the Union Square territory in the upcoming steps.
import folium
from branca.element import Element
import geopandas as gpd
from shapely import wkt

print("🗺️ Converting BigQuery geography data for mapping...")
# 1. Parse WKT to Geometry
nyc_h3_df['geometry'] = nyc_h3_df['geography'].apply(wkt.loads)

# 2. Create the GeoDataFrame
gdf_h3 = gpd.GeoDataFrame(nyc_h3_df, geometry='geometry', crs="EPSG:4326")

if gdf_h3.empty:
    raise ValueError("No H3 data found! Please ensure Cell 2 ran successfully.")

# --- THE FIX ---
# Drop the 'sample_place_ids' (ndarray) and raw 'geography' columns
# so Folium can successfully serialize the data to JSON.
gdf_h3_map = gdf_h3[['h3_cell_index', 'count', 'geometry']].copy()
df_h3_map = nyc_h3_df[['h3_cell_index', 'count']].copy()
# ---------------

# 3. Initialize Map using Google Maps Backend Assets
m_hex = folium.Map(
    location=[CITY_CENTER_LAT, CITY_CENTER_LNG],
    zoom_start=11,
    tiles=tiles_template_url,
    attr=google_attribution,
    name="Google Maps",
    control_scale=True
)

# 4. Add Choropleth Layer
folium.Choropleth(
    geo_data=gdf_h3_map,      # Using the cleaned GeoDataFrame
    data=df_h3_map,           # Using the cleaned Pandas DataFrame
    columns=['h3_cell_index', 'count'],
    key_on='feature.properties.h3_cell_index',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.1,
    legend_name='Place Density (Target Businesses per H3 Cell)'
).add_to(m_hex)

# 5. Inject Google Logo
m_hex.get_root().html.add_child(Element(logo_html))

print("✅ Rendering Heatmap...")
display(m_hex)

### Normalize your CRM Data to Place IDs

Before we can find "net-new" leads, we must translate our raw CRM text records into definitive Google Maps **Place IDs**. If we try to run a BigQuery `LEFT JOIN` exclusion using unstructured text strings, the match rates will be exceptionally poor.

As detailed in the [Places Insights Sales Leads Architecture](https://developers.google.com/maps/architecture/places-insights-sales-leads#step_3_normalize_your_crm_data_to_place_ids), we use a smart, cost-effective API pipeline to cleanse this data:
1. **Address Validation API (The First Check):** We test if the raw address resolves directly to an *Establishment* Place ID.
2. **Text Search (New) API (The Fallback):** If the validation only returns the building (premise), we extract the newly cleansed address and combine it with the business name to pinpoint the exact establishment ID.

> **💡 Production Best Practice:**
> In this Colab notebook, we are using the `requests` library to demonstrate the API logic without triggering dependency conflicts. However, when migrating this pipeline to a **production environment** (such as Cloud Functions or a dedicated data engineering pipeline), it is highly recommended to use the Google Maps Platform Python Client Libraries ([`google-maps-addressvalidation`](https://developers.google.com/maps/documentation/address-validation/client_libraries) and [`google-maps-places`](https://developers.google.com/maps/documentation/places/web-service/client-libraries)). The client libraries provide automatic retry logic for rate limits (HTTP 429s), built-in gRPC performance benefits, and strict type safety.

In [ ]:
# @title 6. Normalize CRM Data to Place IDs
# @markdown Run raw CRM text through Address Validation and Text Search (New) to extract definitive Establishment Place IDs.
import pandas as pd
import requests

# 1. Create the Mock CRM Data (from the architecture document)
crm_data =[
    {"Place Name": "Boucherie Union Square", "Address": "225 Park Ave S, New York, NY 10003, United States"},
    {"Place Name": "Gramercy Tavern", "Address": "42 E 20th St, New York, NY 10003, United States"},
    {"Place Name": "Barn Joo Union Square", "Address": "35 Union Square W, New York, NY 10003, United States"},
    {"Place Name": "LOS TACOS No.1", "Address": "200 Park Ave S, New York, NY 10003, United States"},
    {"Place Name": "Union Square Cafe", "Address": "101 E 19th St, New York, NY 10003, United States"}
]
df_crm = pd.DataFrame(crm_data)

print("🚀 Starting CRM Normalization Pipeline...\n")

normalized_records =[]

# 2. Iterate through the CRM records
for index, row in df_crm.iterrows():
    name = row['Place Name']
    raw_address = row['Address']
    print(f"🔄 Processing: {name}")

    place_id = None
    match_type = "Failed to Match"

    # --- STEP A: Address Validation API ---
    av_url = f"https://addressvalidation.googleapis.com/v1:validateAddress?key={GMP_API_KEY}"
    av_payload = {
        "address": {
            "regionCode": "US",
            "addressLines":[f"{name}, {raw_address}"] # Passing Name + Address
        }
    }

    av_response = requests.post(av_url, json=av_payload).json()

    # Safely extract geocode results
    if 'result' in av_response and 'geocode' in av_response['result']:
        place_types = av_response['result']['geocode'].get('placeTypes',[])

        # Check if it resolved down to the establishment level
        if 'establishment' in place_types or 'point_of_interest' in place_types:
            place_id = av_response['result']['geocode'].get('placeId')
            match_type = "Address Validation API"
            print(f"   ✅ Exact business found via Address Validation!")
        else:
            # --- STEP B: Text Search (New) API (Fallback) ---
            # It only found the building. We grab the cleansed address and query Text Search.
            cleansed_address = av_response['result']['address'].get('formattedAddress', raw_address)
            print(f"   ⚠️ Only building found. Falling back to Text Search with address: {cleansed_address}")

            ts_url = "https://places.googleapis.com/v1/places:searchText"
            ts_headers = {
                "X-Goog-Api-Key": GMP_API_KEY,
                "X-Goog-FieldMask": "places.id", # Crucial: Only request the ID to minimize costs
                "Content-Type": "application/json"
            }
            ts_payload = {
                "textQuery": f"{name} at {cleansed_address}"
            }

            ts_response = requests.post(ts_url, headers=ts_headers, json=ts_payload).json()

            if 'places' in ts_response and len(ts_response['places']) > 0:
                place_id = ts_response['places'][0].get('id')
                match_type = "Text Search (New) API"
                print(f"   ✅ Establishment found via Text Search!")

    # Append results
    normalized_records.append({
        "Place Name": name,
        "Original Address": raw_address,
        "Place ID": place_id,
        "Match Type": match_type
    })

# 3. Create and display the final normalized DataFrame
df_normalized = pd.DataFrame(normalized_records)
print("\n🎉 Pipeline Complete! Here is your normalized CRM data ready for BigQuery:")
display(df_normalized)

### Perform Whitespace Exclusion Analysis in BigQuery

Now that our CRM records are accurately mapped to Place IDs, we can perform the actual whitespace analysis. This step leverages BigQuery's spatial functions and standard SQL to cross-reference our CRM data against the Google Maps Places Insights dataset.

As outlined in the [Places Insights Sales Leads Architecture](https://developers.google.com/maps/architecture/places-insights-sales-leads#step_4_perform_whitespace_exclusion_analysis_in_bigquery), the logic relies on a BigQuery `LEFT JOIN`:

1. **Extract Territory:** We query the `PLACES_COUNT_PER_H3` function to get all operational target businesses in our chosen sales territory.
2. **Flatten Arrays:** We `UNNEST` the resulting `sample_place_ids` array so every Google Maps business gets its own individual row.
3. **The Exclusion Join:** We `LEFT JOIN` this massive list against the small list of normalized CRM Place IDs we generated in the previous step.
4. **Identify the Whitespace:** By checking the results of the join, we can flag existing customers (as a "Proof Output") or filter them out completely to generate a list of 100% **net-new sales leads** with actionable Google Maps URLs.

In [ ]:
# @title 7. Perform Whitespace Exclusion Analysis in BigQuery
# @markdown Run a spatial query around Union Square and flag existing CRM matches vs. Net-New Leads.

# 1. Extract the successfully matched Place IDs from our Pandas DataFrame
crm_place_ids = df_normalized['Place ID'].dropna().tolist()

# Format the Python list into a comma-separated SQL string (e.g., "'ChIJ...', 'ChIJ...'")
formatted_crm_ids = ", ".join([f"'{pid}'" for pid in crm_place_ids])

print(f"🔗 Injecting {len(crm_place_ids)} normalized CRM Place IDs into BigQuery...")

# 2. Construct the Whitespace Exclusion Query (Proof Version)
# Notice we are targeting the Union Square area with an 850m radius and a tight h3_resolution of 10.
exclusion_query = f"""
WITH existing_customers AS (
  -- 1. Upload the CRM Place IDs
  SELECT * FROM UNNEST([{formatted_crm_ids}]) AS place_id
),
target_area_businesses AS (
  -- 2. Query Places Insights for all operational targets around Union Square
  SELECT h3_cell_index, place_id
  FROM `{GCP_PROJECT_ID}.places_insights___us.PLACES_COUNT_PER_H3`(
    JSON_OBJECT(
      'geography', ST_GEOGPOINT(-73.99043, 40.73595),
      'geography_radius', 850,
      'h3_resolution', 10,
      'types',['restaurant', 'bar', 'cafe', 'coffee_shop'],
      'business_status', ['OPERATIONAL']
    )
  ), UNNEST(sample_place_ids) AS place_id
)
-- 3. The "Proof" Output: Join and Flag
SELECT
  t.h3_cell_index,
  t.place_id,
  CASE
    WHEN e.place_id IS NOT NULL THEN 'Existing Customer (To Be Excluded)'
    ELSE 'Net-New Lead'
  END AS lead_status,
  CONCAT('https://www.google.com/maps/search/?api=1&query=Place&query_place_id=', t.place_id) AS actionable_maps_url
FROM target_area_businesses t
LEFT JOIN existing_customers e ON t.place_id = e.place_id
ORDER BY
  -- Explicitly sort the existing customers to the top (0 comes before 1)
  CASE WHEN e.place_id IS NOT NULL THEN 0 ELSE 1 END ASC;
"""

try:
    print("🚀 Running BigQuery Spatial JOIN...")
    # Execute query and load into Pandas
    leads_df = client.query(exclusion_query).to_dataframe()

    print(f"✅ Query complete! Evaluated {len(leads_df)} businesses.")
    display(leads_df.head(10))

except Exception as e:
    print(f"❌ Query failed: {e}")

### Moving to Production: Generating the Final Net-New Lead List

As we can see in the table above, our `LEFT JOIN` logic successfully identified our 5 existing CRM customers out of the hundreds of businesses in the Union Square territory!

However, when handing a list over to a sales team, we don't want them to see existing customers at all. We want a clean list of 100% untouched prospects.

To turn this into a final [Production Query](https://developers.google.com/maps/architecture/places-insights-sales-leads#production_query), you simply strip out the "Proof" `CASE` statements and add a strict `WHERE` filter to drop the matches.

The final, production-ready `SELECT` statement at the bottom of your BigQuery script should look like this:

```sql
SELECT
  t.h3_cell_index,
  t.place_id,
  CONCAT('https://www.google.com/maps/search/?api=1&query=Place&query_place_id=', t.place_id) AS actionable_maps_url
FROM target_area_businesses t
LEFT JOIN existing_customers e ON t.place_id = e.place_id
WHERE e.place_id IS NULL; -- 🚨 The Production Exclusion Filter
```

By applying `WHERE e.place_id IS NULL`, BigQuery completely discards any Google Maps business that already exists in your CRM table, leaving you with a highly targeted, whitespace-only lead list! You can then use Pandas to export this directly to a CSV: `leads_df.to_csv('net_new_leads.csv', index=False)`.